# Verify Diffusion Vocab From Sample Data

This notebook helps answer a narrow question:

Can the diffusion checkpoint at `checkpoints/sample_large_diffusion_refiner/last.ckpt` be paired with a vocab rebuilt from the sample token dataset already in this repo?

It does **not** mutate the repo. It only:

- loads the diffusion checkpoint
- inspects its output vocabulary size from the model weights
- rebuilds notebook-style vocab candidates from sample token JSON files
- compares candidate vocab sizes and token inventories against the checkpoint shape
- optionally runs a dry compatibility check using the packaged refiner class


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import torch

REPO_ROOT = Path.cwd()
CHECKPOINT_PATH = REPO_ROOT / "checkpoints" / "sample_large_diffusion_refiner" / "last.ckpt"
SAMPLE_TOKEN_DIR = REPO_ROOT / "sample_data_large" / "beat_aligned_dataset" / "token_json"

print(f"repo_root={REPO_ROOT}")
print(f"checkpoint_exists={CHECKPOINT_PATH.exists()}")
print(f"sample_token_dir_exists={SAMPLE_TOKEN_DIR.exists()}")


repo_root=c:\Users\28548\PythonNotebooks\taiko-diffusion
checkpoint_exists=True
sample_token_dir_exists=True


In [2]:
def load_checkpoint_payload(path: Path) -> dict[str, Any]:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(payload, dict):
        raise TypeError(f"Expected dict checkpoint payload, got {type(payload)!r}")
    return payload


def checkpoint_vocab_size_from_state_dict(payload: dict[str, Any]) -> int:
    state = payload["model_state_dict"]
    output_weight = state["output_layer.weight"]
    token_embed_weight = state["token_embed.weight"]
    if output_weight.shape[0] != token_embed_weight.shape[0]:
        raise ValueError(
            f"Output/embed vocab mismatch: output={tuple(output_weight.shape)} embed={tuple(token_embed_weight.shape)}"
        )
    return int(output_weight.shape[0])


def summarize_checkpoint(payload: dict[str, Any]) -> None:
    state = payload["model_state_dict"]
    print("top_level_keys=", sorted(payload.keys()))
    print("epoch=", payload.get("epoch"))
    print("has_architecture_spec=", "architecture_spec" in payload)
    print("has_vocab=", "vocab" in payload)
    print("output_layer.weight.shape=", tuple(state["output_layer.weight"].shape))
    print("output_layer.bias.shape=", tuple(state["output_layer.bias"].shape))
    print("token_embed.weight.shape=", tuple(state["token_embed.weight"].shape))
    print("input_proj.weight.shape=", tuple(state["input_proj.weight"].shape))
    print("pos_encoder.shape=", tuple(state["pos_encoder"].shape))
    print("pos_decoder.shape=", tuple(state["pos_decoder"].shape))


In [3]:
payload = load_checkpoint_payload(CHECKPOINT_PATH)
summarize_checkpoint(payload)
CHECKPOINT_VOCAB_SIZE = checkpoint_vocab_size_from_state_dict(payload)
print(f"checkpoint_vocab_size={CHECKPOINT_VOCAB_SIZE}")


top_level_keys= ['epoch', 'model_state_dict', 'optimizer_state_dict']
epoch= 9
has_architecture_spec= False
has_vocab= False
output_layer.weight.shape= (124, 256)
output_layer.bias.shape= (124,)
token_embed.weight.shape= (124, 256)
input_proj.weight.shape= (256, 128)
pos_encoder.shape= (1, 2048, 256)
pos_decoder.shape= (1, 2048, 256)
checkpoint_vocab_size=124


## Rebuild notebook-style candidate vocabs

The diffusion notebook builds vocab from token JSON files using this rule:

1. collect all tokens appearing in all split files
2. prepend special tokens
3. append sorted event tokens
4. append sorted `TS_*` tokens

We test two candidates here:

- AR-style specials: `PAD`, `BOS`, `EOS`
- diffusion-style specials: `PAD`, `BOS`, `EOS`, `MASK`


In [4]:
def load_all_tokens_from_token_dir(token_dir: Path) -> set[str]:
    token_set: set[str] = set()
    json_paths = sorted(token_dir.glob("*.json"))
    if not json_paths:
        raise FileNotFoundError(f"No token JSON files found under {token_dir}")

    for path in json_paths:
        rows = json.loads(path.read_text(encoding="utf-8"))
        for row in rows:
            for token in row.get("tokens", []):
                token_set.add(str(token))
    return token_set


def build_vocab_from_token_set(token_set: set[str], special_tokens: list[str]) -> dict[str, Any]:
    event_tokens = sorted(token for token in token_set if not token.startswith("TS_"))
    ts_tokens = sorted(
        (token for token in token_set if token.startswith("TS_")),
        key=lambda token: int(token.split("_", 1)[1]),
    )
    vocab_list = list(special_tokens) + event_tokens + ts_tokens
    token_to_id = {token: idx for idx, token in enumerate(vocab_list)}
    id_to_token = {idx: token for token, idx in token_to_id.items()}
    return {
        "special_tokens": list(special_tokens),
        "event_tokens": event_tokens,
        "ts_tokens": ts_tokens,
        "vocab_list": vocab_list,
        "token_to_id": token_to_id,
        "id_to_token": id_to_token,
    }


token_set = load_all_tokens_from_token_dir(SAMPLE_TOKEN_DIR)
print(f"sample_token_set_size={len(token_set)}")
print("sample_event_tokens=", sorted(token for token in token_set if not token.startswith("TS_")))
print("sample_ts_range=", (
    min((int(token.split('_', 1)[1]) for token in token_set if token.startswith('TS_')), default=None),
    max((int(token.split('_', 1)[1]) for token in token_set if token.startswith('TS_')), default=None),
))

candidate_ar = build_vocab_from_token_set(token_set, ["PAD", "BOS", "EOS"])
candidate_diffusion = build_vocab_from_token_set(token_set, ["PAD", "BOS", "EOS", "MASK"])

print(f"candidate_ar_vocab_size={len(candidate_ar['vocab_list'])}")
print(f"candidate_diffusion_vocab_size={len(candidate_diffusion['vocab_list'])}")
print(f"candidate_diffusion_matches_checkpoint={len(candidate_diffusion['vocab_list']) == CHECKPOINT_VOCAB_SIZE}")


sample_token_set_size=66
sample_event_tokens= ['BIGDON', 'BIGKAT', 'DON', 'DRUMROLL', 'KAT', 'SLIDEREND', 'SLIDERSTART']
sample_ts_range= (3, 180)
candidate_ar_vocab_size=69
candidate_diffusion_vocab_size=70
candidate_diffusion_matches_checkpoint=False


In [5]:
def compare_candidate_to_checkpoint(candidate: dict[str, Any], checkpoint_vocab_size: int) -> dict[str, Any]:
    vocab_size = len(candidate["vocab_list"])
    return {
        "special_tokens": candidate["special_tokens"],
        "vocab_size": vocab_size,
        "delta_vs_checkpoint": int(checkpoint_vocab_size - vocab_size),
        "has_mask": "MASK" in candidate["token_to_id"],
        "event_count": len(candidate["event_tokens"]),
        "ts_count": len(candidate["ts_tokens"]),
        "max_ts": max((int(token.split('_', 1)[1]) for token in candidate["ts_tokens"]), default=None),
    }


comparison_rows = [
    compare_candidate_to_checkpoint(candidate_ar, CHECKPOINT_VOCAB_SIZE),
    compare_candidate_to_checkpoint(candidate_diffusion, CHECKPOINT_VOCAB_SIZE),
]
comparison_rows


[{'special_tokens': ['PAD', 'BOS', 'EOS'],
  'vocab_size': 69,
  'delta_vs_checkpoint': 55,
  'has_mask': False,
  'event_count': 7,
  'ts_count': 59,
  'max_ts': 180},
 {'special_tokens': ['PAD', 'BOS', 'EOS', 'MASK'],
  'vocab_size': 70,
  'delta_vs_checkpoint': 54,
  'has_mask': True,
  'event_count': 7,
  'ts_count': 59,
  'max_ts': 180}]

## Optional dry model compatibility check

If a candidate vocab size matches the checkpoint exactly, this cell tries to instantiate the packaged refiner class and load the weights.

If it does not match, the cell will stop early and explain why.

In [6]:
CANDIDATE_NAME = "candidate_diffusion"
candidate = {
    "candidate_ar": candidate_ar,
    "candidate_diffusion": candidate_diffusion,
}[CANDIDATE_NAME]

if len(candidate["vocab_list"]) != CHECKPOINT_VOCAB_SIZE:
    print(
        "Skipping model load: candidate vocab size "
        f"{len(candidate['vocab_list'])} does not match checkpoint vocab size {CHECKPOINT_VOCAB_SIZE}."
    )
else:
    from src.model.diffusion_refiner import TaikoDiffusionRefiner

    model = TaikoDiffusionRefiner(
        vocab_size=len(candidate["vocab_list"]),
        input_dim=128,
        d_model=256,
        nhead=4,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=1024,
        dropout=0.3,
        max_len=2048,
    )
    missing, unexpected = model.load_state_dict(payload["model_state_dict"], strict=False)
    print("missing_keys=", missing)
    print("unexpected_keys=", unexpected)
    x_audio = torch.randn(1, 1536, 128)
    x_tokens = torch.randint(low=0, high=len(candidate["vocab_list"]), size=(1, 32))
    x_attention = torch.ones_like(x_tokens)
    logits = model(x_audio, x_tokens, decoder_attention_mask=x_attention)
    print("forward_logits_shape=", tuple(logits.shape))


Skipping model load: candidate vocab size 70 does not match checkpoint vocab size 124.


## Interpretation guide

- If `candidate_diffusion_vocab_size` does **not** equal `checkpoint_vocab_size`, then the sample-data vocab is not enough.
- If sizes match, that is only a first gate. You should still treat the token mapping as tentative unless it came from the same diffusion training corpus.
- If sizes do not match, the missing information is the original diffusion token dataset or a saved diffusion `vocab.json`.
